# Agent Config API Demo — TypeScript / Deno

This notebook mirrors the Python `agent_config_demo.ipynb` using the TypeScript SDK
and Deno's Jupyter kernel (`deno jupyter`).

Topics covered:

1. `getOrCreateConfig` — first call auto-creates from fallback
2. `getOrCreateConfig` — returns fallback values when backend is unreachable
3. `createConfig` — unconditionally write a new version
4. `setConfigEnv` — tag a version with an environment name
5. Fetch by `env` and by explicit `version` name
6. `getOrCreateConfig` without a fallback (generic return type)
7. `agentConfigContext` — mask + blueprint pinning

> **Prerequisites**: a running Opik backend. Set `OPIK_URL_OVERRIDE` (or rely on
> the default `http://localhost:5173`) and optionally `OPIK_API_KEY`.

## Setup

In [1]:
// Deno imports — npm: specifier resolves from the npm registry.
import { Opik, track, agentConfigContext } from "npm:opik";
import { getGlobalBlueprintRegistry } from "npm:opik/agent-config/blueprintCache";

TypeError: Import "async_hooks" not a dependency
  [0m[36mhint:[0m If you want to use a built-in Node module, add a "node:" prefix (ex. "node:async_hooks").
    at [0m[36mfile:///Users/petrot/Development/opik/sdks/typescript/dist/index.js[0m:[0m[33m1[0m:[0m[33m525[0m

In [2]:
const client = new Opik({
  // Reads OPIK_URL_OVERRIDE + OPIK_API_KEY from env automatically.
  // Pass { url, apiKey } here to override explicitly.
});

const PROJECT = `agent-config-demo-${Date.now()}-${Math.random().toString(36).slice(2, 7)}`;
console.log("Project:", PROJECT);

// Cleanup helper — called at the end of the notebook.
async function deleteProject(name: string): Promise<void> {
  try {
    const project = await client.api.projects.retrieveProject({ name });
    if (project?.id) {
      await client.api.projects.deleteProjectById(project.id);
      console.log(`Deleted project "${name}"`);
    }
  } catch {
    // best effort
  }
}

ReferenceError: Opik is not defined

---
## 1. `getOrCreateConfig` — first call auto-creates from fallback

The project is empty. `getOrCreateConfig` detects this and writes the fallback
values as the first version. The backend automatically tags the first version as
`"prod"`.

> `getOrCreateConfig` **must** be called inside a `track()` function.

In [ ]:
const fallbackV1 = {
  temperature: 0.5,
  model: "gpt-3.5-turbo",
  system_prompt: "You are a helpful assistant.",
};

const cfgV1 = await track({ projectName: PROJECT }, async () =>
  client.getOrCreateConfig({ fallback: fallbackV1, projectName: PROJECT })
)();

console.log("isFallback  :", cfgV1.isFallback);    // false — stored in backend
console.log("temperature :", cfgV1.temperature);
console.log("model       :", cfgV1.model);
console.log("system_prompt:", cfgV1.system_prompt);

console.assert(cfgV1.isFallback === false, "auto-created config must not be marked as fallback");

---
## 2. `getOrCreateConfig` — returns fallback when backend is unreachable

When the backend times out or cannot be reached **and** a `fallback` is provided,
`getOrCreateConfig` returns the fallback with `isFallback: true` instead of
throwing. We simulate this by pointing a second client at a non-existent host.

In [ ]:
const unreachableClient = new Opik({
  url: "http://127.0.0.1:19999", // nothing listening here
  apiKey: "demo",
});

const offlineCfg = await track({ projectName: PROJECT }, async () =>
  unreachableClient.getOrCreateConfig({
    fallback: fallbackV1,
    projectName: PROJECT,
  })
)();

console.log("isFallback  :", offlineCfg.isFallback);  // true
console.log("temperature :", offlineCfg.temperature);  // from local fallback
console.log("model       :", offlineCfg.model);

console.assert(offlineCfg.isFallback === true, "offline cfg must be marked as fallback");

---
## 3. `createConfig` — write a new version unconditionally

`createConfig` does **not** require a `track()` context and always creates a new
version. It returns the version name string used in subsequent calls.

In [ ]:
const v2Name = await client.createConfig(
  {
    temperature: 0.8,
    model: "gpt-4o",
    system_prompt: "You are an expert assistant. Think step by step.",
  },
  { projectName: PROJECT, description: "Upgraded to gpt-4o with chain-of-thought prompt" }
);

console.log("Created version:", v2Name);
console.assert(typeof v2Name === "string" && v2Name.length > 0);

---
## 4. `setConfigEnv` — tag a version with an environment

`"prod"` currently points to the v1 values (auto-tagged on first write). Promote
v2 to `"prod"` and additionally tag it as `"staging"`.

In [ ]:
await client.setConfigEnv({ version: v2Name, env: "prod", projectName: PROJECT });
console.log(`Version "${v2Name}" tagged as 'prod'`);

await client.setConfigEnv({ version: v2Name, env: "staging", projectName: PROJECT });
console.log(`Version "${v2Name}" tagged as 'staging'`);

---
## 5. Fetch by `env` — confirm prod now returns v2 values

In [ ]:
// Clear the local blueprint cache so we hit the backend, not a cached copy.
getGlobalBlueprintRegistry().clear();

const prodCfg = await track({ projectName: PROJECT }, async () =>
  client.getOrCreateConfig({ fallback: fallbackV1, projectName: PROJECT, env: "prod" })
)();

console.log("=== env='prod' (should be v2 after setConfigEnv) ===");
console.log("temperature  :", prodCfg.temperature);
console.log("model        :", prodCfg.model);
console.log("system_prompt:", prodCfg.system_prompt);

console.assert(Math.abs(prodCfg.temperature - 0.8) < 1e-9);
console.assert(prodCfg.model === "gpt-4o");

---
## 6. Fetch by explicit `version` name

In [ ]:
getGlobalBlueprintRegistry().clear();

const byNameCfg = await track({ projectName: PROJECT }, async () =>
  client.getOrCreateConfig({ fallback: fallbackV1, projectName: PROJECT, version: v2Name })
)();

console.log(`=== Fetch by version "${v2Name}" ===`);
console.log("temperature:", byNameCfg.temperature);
console.log("model      :", byNameCfg.model);
console.log("blueprintVersion:", byNameCfg.blueprintVersion);

console.assert(byNameCfg.blueprintVersion === v2Name);
console.assert(Math.abs(byNameCfg.temperature - 0.8) < 1e-9);

---
## 7. Fetch without a fallback (generic `Config` return type)

Omitting `fallback` returns a generic `Config<Record<string, unknown>>`. If the
project is empty this throws `ConfigNotFoundError` instead of auto-creating.

In [ ]:
getGlobalBlueprintRegistry().clear();

const noFallbackCfg = await track({ projectName: PROJECT }, async () =>
  client.getOrCreateConfig({ projectName: PROJECT })
)();

console.log("=== No fallback ===");
console.log("isFallback      :", noFallbackCfg.isFallback);  // false
console.log("temperature     :", noFallbackCfg.temperature);
console.log("model           :", noFallbackCfg.model);
console.log("blueprintId     :", noFallbackCfg.blueprintId);

// Must resolve to the same underlying blueprint as the explicit-version fetch.
console.assert(noFallbackCfg.isFallback === false);
console.assert(noFallbackCfg.blueprintId === byNameCfg.blueprintId);
console.assert(noFallbackCfg.temperature === byNameCfg.temperature);
console.assert(noFallbackCfg.model === byNameCfg.model);

---
## 8. `agentConfigContext` — mask and blueprint pinning

An `agentConfigContext` lets you override which blueprint version or mask is active
for all `getOrCreateConfig` calls within the async scope — useful for A/B testing
or per-request overrides without touching the backend state.

In [ ]:
import { ConfigManager } from "npm:opik/agent-config";

// Publish a second named version so we have something to pin.
const v3Name = await client.createConfig(
  { temperature: 0.7, model: "gpt-4o-mini", system_prompt: "Be concise." },
  { projectName: PROJECT }
);
await client.setConfigEnv({ version: v2Name, env: "prod", projectName: PROJECT });

// Create a mask that overrides only temperature.
const manager = new ConfigManager(PROJECT, client);
const maskId = await manager.createMask({
  values: [{ key: "temperature", value: "0.99", type: "float" }],
});
console.log("maskId:", maskId);

const fetchCfg = track({ projectName: PROJECT }, () =>
  client.getOrCreateConfig({ fallback: fallbackV1, projectName: PROJECT })
);

In [ ]:
// Baseline: no context → prod (v2) values.
getGlobalBlueprintRegistry().clear();
const baseline = await fetchCfg();
console.log("Baseline — temperature:", baseline.temperature, "model:", baseline.model);
console.assert(Math.abs(baseline.temperature - 0.8) < 1e-9);
console.assert(baseline.model === "gpt-4o");

In [ ]:
// Mask only: overlays temperature=0.99 on top of prod (v2); model stays "gpt-4o".
getGlobalBlueprintRegistry().clear();
let withMask: Awaited<ReturnType<typeof fetchCfg>>;
await agentConfigContext({ maskId }, async () => {
  withMask = await fetchCfg();
});
console.log("Mask only — temperature:", withMask!.temperature, "model:", withMask!.model);
console.assert(Math.abs(withMask!.temperature - 0.99) < 1e-9);
console.assert(withMask!.model === "gpt-4o");

In [ ]:
// Blueprint pin: switch to v3 without a mask → v3 values.
getGlobalBlueprintRegistry().clear();
let pinnedV3: Awaited<ReturnType<typeof fetchCfg>>;
await agentConfigContext({ blueprintName: v3Name }, async () => {
  pinnedV3 = await fetchCfg();
});
console.log("Pinned v3 — temperature:", pinnedV3!.temperature, "model:", pinnedV3!.model);
console.assert(Math.abs(pinnedV3!.temperature - 0.7) < 1e-9);
console.assert(pinnedV3!.model === "gpt-4o-mini");

In [ ]:
// Blueprint + mask: pin v3 AND apply mask → temperature overridden, model from v3.
getGlobalBlueprintRegistry().clear();
let pinnedAndMasked: Awaited<ReturnType<typeof fetchCfg>>;
await agentConfigContext({ blueprintName: v3Name, maskId }, async () => {
  pinnedAndMasked = await fetchCfg();
});
console.log("Pinned v3 + mask — temperature:", pinnedAndMasked!.temperature, "model:", pinnedAndMasked!.model);
console.assert(Math.abs(pinnedAndMasked!.temperature - 0.99) < 1e-9);
console.assert(pinnedAndMasked!.model === "gpt-4o-mini");

---
## Cleanup

In [ ]:
await deleteProject(PROJECT);